# Data Loading Demonstration


In [ ]:
%pip install pandas pyarrow tqdm

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import sys
sys.path.append('./lib')

## Loading Triage Malware Dataset

The dataset contains annotated TLS communication samples collected from a Triage-based malware sandbox environment.  
Each sample corresponds to network traffic generated during the execution of a malware specimen under controlled conditions.

The following columns represent the relevant semantic annotations used for analysis and modeling:

- **meta.application** — Application identifier. This field is not used in this dataset and is therefore set to `NA`.
- **meta.system** — Operating system metadata associated with the communication, typically indicating the OS family and version of the sandboxed environment.
- **meta.service** — Network-facing service or operating system subsystem derived from system metadata, enabling coarse-grained service-level classification.
- **meta.malware.family** — Malware family annotation. Samples without a confidently identified family are labeled as `NA`.
- **meta.host** — Host-level identifier or role. This field is not used in this dataset and is therefore set to `NA`.

Note that a single network connection may be annotated both with a malware family and a system service.  
In such cases, the connection is *triggered by the execution of the malware sample*, but the observed TLS communication corresponds to a **legitimate operating system service** (e.g., update checks, certificate validation, or telemetry) invoked by the malware rather than to a malware-specific network protocol.


## Load all data

In [ ]:
from data_helper import get_families_frequency
import pyarrow.dataset as ds
import pyarrow.compute as pc

ds_malware = ds.dataset(f"../datasets/malware.parquet", format="parquet")


df_malware = ds_malware.to_table().to_pandas()
get_families_frequency(df_malware)

## Loading per family
The dataset allows you to load only the columns you need, so you can collect information on available families without loading the entire dataset.
Then, the individual family datasets are loaded.

In [ ]:
# Collect available familes:
families = (
    ds_malware
    .to_table(columns=["meta.malware.family"])
    .column("meta.malware.family")
    .unique()
    .to_pylist()
)
print(families)
# Load familes one by one:
for family in families:
    if family is not None:
        df_family = ds_malware.to_table(filter=(pc.field("meta.malware.family") == family)).to_pandas()
        print(f"{family} - {df_family.shape[0]} / {len(df_family)}")

It is possible to do more data analysis without reading the entire dataset using dataset. For instance, we can compute the count of samples for every malware family.

In [ ]:
freq = (
    ds_malware
    .to_table(columns=["meta.malware.family"])
    .to_pandas()
    ["meta.malware.family"]
    .value_counts()
    .sort_index()
)
freq

It is often best to load only the necessary columns and then filter or compute on the data frame.

In [ ]:
_df = (
    ds_malware
    .to_table(columns=["meta.malware.family"])
    .to_pandas())
non_malware_samples = _df["meta.malware.family"].isna().sum()
print(f"Non malware samples={non_malware_samples}")

## Loading Windows Applications Dataset

The dataset contains annotated TLS communication samples collected from applications executed in a controlled Windows environment.  
Traffic was generated by legitimate Windows-based applications running inside **Windows Sandbox**, ensuring a clean and reproducible execution context.

The following columns represent the relevant semantic annotations used for analysis and modeling:

- **meta.application** — Application identifier in the normalized form `VENDOR.APPLICATION`, representing the originating software (e.g., `MICROSOFT.OUTLOOK`, `GOOGLE.CHROME`).
- **meta.system** — Operating system metadata associated with the communication, indicating the OS family and version of the sandboxed environment.
- **meta.service** — Network-facing service annotation. This field is not used in this dataset and is set to `NA`.
- **meta.malware** — Malware family annotation. This field is not used in this dataset and is set to `NA`.
- **meta.host** — Host-level identifier or role. This field is not used in this dataset and is set to `NA`.

The dataset includes only TLS connections directly attributable to the executed applications.  
Background operating system traffic and unrelated connections were explicitly filtered out to preserve a clean, application-centric view of network behavior.

In [ ]:
from data_helper import get_applications_frequency
import pyarrow.dataset as ds
import pyarrow.compute as pc

dataset = ds.dataset(f"../datasets/winapps.parquet", format="parquet")
print("Loading dataset...")
df_applications = dataset.to_table().to_pandas()

get_applications_frequency(df_applications)

# Datasets

## 	D1: WINAPP-APP
Labeled benign reference dataset consisting of TLS connections generated by known Windows applications in a real environment. Serves as the in-distribution baseline for training and calibration.

In [ ]:
import pyarrow.dataset as ds

dataset = ds.dataset(f"../datasets/winapps.parquet", format="parquet")
print("Loading dataset...")
df_winapp_app = dataset.to_table().to_pandas()

## D2: TRIAGE-MAL24

Labeled malware dataset (2024) containing TLS connections produced by malware samples executed in a sandbox environment. Used for semantic OOD evaluation without temporal shift.

In [ ]:
import pyarrow.dataset as ds
import pyarrow.compute as pc

ds_malware = ds.dataset(f"../datasets/malware.parquet", format="parquet")

filt = (
    pc.match_like(pc.field("meta.sample.id"), "24%") &  # prefix match
    pc.is_valid(pc.field("meta.malware.family"))        # non-null check
)

df_triage_mal24 = ds_malware.to_table(
    filter=filt
).to_pandas()

df_triage_mal24


## D3: TRIAGE-MAL25

Labeled malware dataset (future) representing malware TLS traffic collected after the benign training period, introducing both semantic and temporal distribution shift.

In [ ]:
import pyarrow.dataset as ds
import pyarrow.compute as pc

ds_malware = ds.dataset(f"../datasets/malware.parquet", format="parquet")

filt = (
    pc.match_like(pc.field("meta.sample.id"), "25%") &  # prefix match
    pc.is_valid(pc.field("meta.malware.family"))        # non-null check
)

df_triage_mal25 = ds_malware.to_table(
    filter=filt
).to_pandas()

df_triage_mal25

## D4: TRIAGE-BG

Sandbox background dataset comprising non-malicious TLS connections generated by sandbox infrastructure and auxiliary processes. Used to isolate sandbox-induced domain shift from malicious behavior.

In [ ]:
import pyarrow.dataset as ds
import pyarrow.compute as pc

ds_malware = ds.dataset(f"../datasets/malware.parquet", format="parquet")

filt = (
    pc.match_like(pc.field("meta.sample.id"), "25%") &  # prefix match
    pc.is_null(pc.field("meta.malware.family"))        # non-null check
)

df_triage_bg25 = ds_malware.to_table(
    filter=filt
).to_pandas()

df_triage_bg25

## D5: REAL

Unlabeled real-network dataset capturing TLS connections from an operational network environment. Used to assess deployment realism, unknown variability, and in-the-wild alert rates.

In [15]:
import pyarrow.dataset as ds
import pyarrow.compute as pc

ds_soho = ds.dataset(f"../datasets/soho.parquet", format="parquet")

df_real = ds_soho.to_table().to_pandas()

df_real

,bs,ps,br,pr,dp,sp,da,sa,ts,td,...,tls.ja3,tls.ja4,tls.ja3s,tls.ja4s,meta.sample.id,meta.malware.family,meta.system.os,meta.system.service,meta.application.name,tls.rec
0,4944,15,7574,12,443,64925,20.189.173.9,192.168.1.197,1.758875e+09,3.010000,...,e61d46f1bb7e725311647ad6f8d5e746,t13d1517h2_8daaf6152771_b6f405a00624,15af977ce25de452b96affa2addb1036,t130200_1302_a56c5b993250,None,None,,None,None,"[-1926, 88, 1, -1, -766, 187, 6169, -69, -87, ..."
1,4455,17,5770,16,443,55301,17.248.209.74,192.168.1.197,1.758875e+09,1.673526,...,773906b0efdefa24a7f2b8eb6985bf37,t13d2014h2_a09f3c656075_e42f34c56612,15af977ce25de452b96affa2addb1036,t130200_1302_a56c5b993250,None,None,,None,None,"[-512, 122, 1, 36, 3233, 96, 69, -1, -69, -126..."
2,2514,8,1098,5,443,64929,35.186.234.31,192.168.1.197,1.758875e+09,60.970909,...,4ebbb6c301f067e5c032fc2e839e7e9d,t13d1517h2_8daaf6152771_b6f405a00624,2b0648ab686ee45e0e7c35fcfb0eea7e,t130300_1301_6bbbaf601ed8,None,None,,None,None,"[-2017, 128, 1, 68, -1, -53, 551, 57, 0, 0, 0,..."
3,2942,17,7874,13,443,59984,20.50.88.244,192.168.1.128,1.758875e+09,104.877683,...,02d7359ba777278df8619c1fd64f71e1,t13d121100_a538d02d76d5_f6eeaf393d2e,15af977ce25de452b96affa2addb1036,t130200_1302_a56c5b993250,None,None,,None,None,"[-296, 88, 1, -1, -361, 187, 6077, -69, 98, -5..."
4,3390,13,4897,12,443,64931,140.82.113.25,192.168.1.197,1.758875e+09,2.720050,...,461d8f4ffe2bf1d184efeee502e09585,t13d1516h1_8daaf6152771_0d365e64def3,f4febc55ea12b31ae17cfb7e614afda8,t130200_1301_a56c5b993250,None,None,,None,None,"[-1820, 122, 1, 42, 3151, 96, 53, -1, -53, 74,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108031,2844,11,4992,9,443,60604,142.251.36.74,192.168.1.172,1.759300e+09,1.244736,...,9b02ebd3a43b62d825e1ac605b621dc8,t13d1713h1_5b57614c22b0_eca864cca44a,eb1d94daa7e0344597e756a1fb6e7054,t130200_1301_234ea6891581,None,None,,None,None,"[-512, 122, 1, 3063, -1, -53, -1678, 559, 719,..."
108032,2844,11,5077,10,443,60630,142.251.36.74,192.168.1.172,1.759300e+09,1.244548,...,9b02ebd3a43b62d825e1ac605b621dc8,t13d1713h1_5b57614c22b0_eca864cca44a,eb1d94daa7e0344597e756a1fb6e7054,t130200_1301_234ea6891581,None,None,,None,None,"[-512, 122, 1, 3062, -1, -53, -1678, 559, 753,..."
108033,2844,11,4982,10,443,60616,142.251.36.74,192.168.1.172,1.759300e+09,1.092500,...,9b02ebd3a43b62d825e1ac605b621dc8,t13d1713h1_5b57614c22b0_eca864cca44a,eb1d94daa7e0344597e756a1fb6e7054,t130200_1301_234ea6891581,None,None,,None,None,"[-512, 122, 1, 3062, -1, -53, -1678, 559, 658,..."
108034,5940,38,63596,51,443,58575,23.1.99.35,192.168.1.197,1.759300e+09,1.454849,...,1dc87ef0c4027009d16dee0f5f1bca7f,t13d1517h2_8daaf6152771_b6f405a00624,15af977ce25de452b96affa2addb1036,t130200_1302_a56c5b993250,None,None,,None,None,"[-2017, 122, 1, 46, 3267, 96, 69, -1, -69, -87..."
